# Gaussian Mixture Models — soft, elliptical clustering via EM

> Tutorial pair for [`gaussian_mixture.py`](gaussian_mixture.py).

## 1. Intuition
k-means draws hard, spherical boundaries. A **Gaussian Mixture Model** says the
data was generated by drawing, for each point, a hidden cluster label and then
sampling from that cluster's Gaussian. We don't see the labels, so we infer
*soft* memberships (responsibilities) and fit each cluster's mean **and**
covariance — clusters can be elongated, tilted ellipses of different sizes.

## 2. Concept (the slide)
- **Model:** $p(\mathbf x)=\sum_{k=1}^K \pi_k\,\mathcal N(\mathbf x\mid\boldsymbol\mu_k,\Sigma_k)$
  with mixing weights $\pi_k\ge 0,\ \sum_k\pi_k=1$.
- **Latent variable:** a one-hot $z$ saying which component generated $\mathbf x$.
- **Fit by EM:** the log-likelihood is not concave (sum inside the log), but EM
  alternates a tractable **E-step** (posterior over $z$) and **M-step**
  (weighted Gaussian MLE), each guaranteed not to decrease the likelihood.
- **Covariance type:** *full* (any ellipse) vs *diagonal* (axis-aligned, fewer
  params). k-means $\approx$ GMM with shared spherical $\Sigma=\sigma^2 I$ and
  hardened responsibilities.

## 3. Math derivation

**Incomplete-data log-likelihood** for data $X=\{\mathbf x_i\}$:
$$\ell(\theta)=\sum_{i=1}^n\log\sum_{k=1}^K \pi_k\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k).$$
The $\log\sum$ couples the parameters; direct maximization is hard.

**The ELBO (Jensen lower bound).** Introduce any distribution $q_i(k)$ over the
latent label of point $i$. By Jensen's inequality (log is concave),
$$\ell(\theta)=\sum_i\log\sum_k q_i(k)\frac{\pi_k\mathcal N(\mathbf x_i\mid\theta_k)}{q_i(k)}
\ \ge\ \sum_i\sum_k q_i(k)\log\frac{\pi_k\mathcal N(\mathbf x_i\mid\theta_k)}{q_i(k)}
\ \equiv\ \mathcal L(q,\theta).$$
The gap is exactly $\ell(\theta)-\mathcal L(q,\theta)=\sum_i \mathrm{KL}\!\big(q_i\,\|\,p(z_i\mid\mathbf x_i,\theta)\big)\ge 0$.

**E-step** maximizes $\mathcal L$ over $q$ with $\theta$ fixed: the KL is zero
when $q_i(k)=p(z_i=k\mid\mathbf x_i,\theta)$, the **responsibility**
$$\gamma_{ik}=\frac{\pi_k\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k)}
{\sum_j \pi_j\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_j,\Sigma_j)}.$$
Now the bound *touches* the likelihood: $\mathcal L(q,\theta)=\ell(\theta)$.

**M-step** maximizes $\mathcal L$ over $\theta$ with $q=\gamma$ fixed. Drop the
$-q\log q$ entropy term (constant in $\theta$) and maximize
$Q(\theta)=\sum_i\sum_k \gamma_{ik}\big[\log\pi_k+\log\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k)\big]$.
Setting gradients to zero (with a Lagrange multiplier for $\sum_k\pi_k=1$) gives
the **weighted MLE**, with $N_k=\sum_i\gamma_{ik}$:
$$\boldsymbol\mu_k=\frac1{N_k}\sum_i\gamma_{ik}\mathbf x_i,\quad
\Sigma_k=\frac1{N_k}\sum_i\gamma_{ik}(\mathbf x_i-\boldsymbol\mu_k)(\mathbf x_i-\boldsymbol\mu_k)^\top,\quad
\pi_k=\frac{N_k}{n}.$$

**Why EM monotonically increases $\ell$.** After the E-step,
$\ell(\theta^t)=\mathcal L(q^{t+1},\theta^t)$. The M-step picks $\theta^{t+1}$ to
*maximize* $\mathcal L(q^{t+1},\cdot)$, so $\mathcal L(q^{t+1},\theta^{t+1})\ge\mathcal L(q^{t+1},\theta^t)$.
And $\ell(\theta^{t+1})\ge\mathcal L(q^{t+1},\theta^{t+1})$ because the bound is
a lower bound. Chaining:
$$\ell(\theta^{t+1})\ \ge\ \mathcal L(q^{t+1},\theta^{t+1})\ \ge\ \mathcal L(q^{t+1},\theta^t)\ =\ \ell(\theta^t).$$
So the likelihood never decreases; bounded above, the sequence converges (to a
stationary point — possibly a local max, hence multiple restarts).

**Practical notes.** Compute $\log\mathcal N$ with a Cholesky factor
$\Sigma=LL^\top$ (so $\log\det\Sigma=2\sum_i\log L_{ii}$ and the Mahalanobis term
is $\lVert L^{-1}(\mathbf x-\boldsymbol\mu)\rVert^2$), normalize responsibilities
with log-sum-exp, and add $\varepsilon I$ to $\Sigma_k$ to avoid singular
("collapsing") components.

## 4. NumPy implementation (full + diagonal covariance, BIC selection)

In [ ]:
import inspect, gaussian_mixture as M
for _obj in [M.GMMNumPy]:
    print(inspect.getsource(_obj))

## 5. PyTorch implementation (vectorized closed-form EM, GPU-friendly)

In [ ]:
import inspect, gaussian_mixture as M
for _obj in [M.GMMTorch]:
    print(inspect.getsource(_obj))

## 6. Train / run — ARI, monotone log-likelihood, BIC over K

In [ ]:
import gaussian_mixture as M
M.demo()

## 7. Visualization — soft clusters, covariance ellipses, EM convergence

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.datasets import make_blobs
import gaussian_mixture as M

X, y = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=0)
X = X @ np.array([[0.6, -0.6], [-0.4, 0.8]])   # make blobs anisotropic
gmm = M.GMMNumPy(n_components=3, covariance_type="full").fit(X)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X[:, 0], X[:, 1], c=gmm.labels_, s=10, cmap="tab10")
ax[0].scatter(gmm.means_[:, 0], gmm.means_[:, 1], c="k", marker="X", s=160)
for k in range(gmm.K):                          # draw 2-sigma covariance ellipse
    vals, vecs = np.linalg.eigh(gmm.covariances_[k])
    ang = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    w, h = 2 * 2 * np.sqrt(vals[::-1])
    ax[0].add_patch(Ellipse(gmm.means_[k], w, h, angle=ang, fill=False, edgecolor="k", lw=2))
ax[0].set_title("GMM full covariance (2-sigma ellipses)")

ax[1].plot(gmm.history_, "o-")
ax[1].set_xlabel("EM iteration"); ax[1].set_ylabel("log-likelihood")
ax[1].set_title("EM monotonically increases logL")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- EM only finds a **local** optimum and is init-sensitive → use k-means++ seeding
  and **multiple restarts** (keep the best log-likelihood).
- Without `reg_covar`, a component can collapse onto a single point
  ($\Sigma\to 0$, likelihood $\to\infty$): a singularity, not a real solution.
- **Full** covariance is flexible but $O(d^2)$ params per component; **diagonal**
  is cheaper and robust in high dimensions but only axis-aligned ellipses.
- Choose $K$ with **BIC/AIC** (penalized likelihood), not raw log-likelihood
  (which always improves with more components).

**Next:** drop the probabilistic model and cluster by *density* → DBSCAN.